# Live Trading Tutorial with RL Models

This interactive notebook demonstrates how to deploy trained RL models to live trading on Binance testnet.

## What You'll Learn

1. Load and inspect trained RL models
2. Validate the trading pipeline with historical data
3. Set up safety guards and risk management
4. Run simulated live trading
5. Analyze trading performance

## Prerequisites

- Binance testnet API keys from https://testnet.binance.vision/
- Trained RL model (PPO/A2C/DQN)
- MinIO running with historical data

**IMPORTANT:** This tutorial uses Binance TESTNET with fake money. No real trading occurs.

## Setup

In [1]:
# Imports
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import logging

# Add project to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print(f"✓ Project root: {project_root}")
print(f"✓ Imports complete")

✓ Project root: /Users/mohamedali/trading_project/rl-trading-lab
✓ Imports complete


In [2]:
# Import live trading components
from rl_trading_lab.data import BinanceDataAdapter, BarProcessor, FeaturePipeline
from rl_trading_lab.live import (
    FeatureComputer,
    ModelInferenceEngine,
    OrderExecutor,
    PortfolioManager,
    SafetyGuard,
)
from rl_trading_lab.utils.checkpoint_manager import CheckpointManager

print("✓ Live trading components imported")

✓ Live trading components imported


## Part 1: Load and Inspect Model

First, let's load a trained RL model and inspect its properties.

In [3]:
# Configuration
SYMBOL = "BTCUSDT"

# Discover available models using CheckpointManager
print("Searching for trained models...\n")

available_models = CheckpointManager.discover_all_checkpoints(Path("../checkpoints"))

if available_models:
    print(f"Found {len(available_models)} trained model(s):\n")
    for i, model in enumerate(available_models[:5], 1):  # Show first 5
        print(f"{i}. {model['model_type']} - {model['checkpoint_dir'].name}")
        print(f"   Features: {model['observation_dim']} (observation dimension)")
        print(f"   VecNormalize: {'Yes' if model['vecnormalize_path'] else 'No'}")
    
    if len(available_models) > 5:
        print(f"\n... and {len(available_models) - 5} more models")
    
    # Use the first available model (most recent)
    selected_model = available_models[0]
    MODEL_PATH = str(selected_model['path'])
    VECNORMALIZE_PATH = str(selected_model['vecnormalize_path']) if selected_model['vecnormalize_path'] else None
    EXPECTED_FEATURES = selected_model['observation_dim']
    
    print(f"\n✓ Selected model: {selected_model['checkpoint_dir'].name}")
    print(f"  Type: {selected_model['model_type']}")
    print(f"  Expected features: {EXPECTED_FEATURES}")
    print(f"  Path: {MODEL_PATH}")
    
    print(f"\n⚠️  IMPORTANT: This model expects {EXPECTED_FEATURES} features.")
    print(f"   Make sure your data pipeline produces exactly {EXPECTED_FEATURES} features!")
else:
    print("✗ No trained models found in ../checkpoints/")
    print("\nPlease train a model first:")
    print("  uv run python experiments/train.py")
    MODEL_PATH = None
    VECNORMALIZE_PATH = None
    EXPECTED_FEATURES = None

Searching for trained models...

Found 33 trained model(s):

1. PPO - PPO_returns_20251105_202508
   Features: 87 (observation dimension)
   VecNormalize: Yes
2. A2C - A2C_returns_20251029_193155
   Features: 87 (observation dimension)
   VecNormalize: Yes
3. A2C - A2C_returns_20251029_192831
   Features: 87 (observation dimension)
   VecNormalize: Yes
4. A2C - A2C_returns_20251029_180503
   Features: 87 (observation dimension)
   VecNormalize: Yes
5. A2C - A2C_returns_20251029_175952
   Features: 87 (observation dimension)
   VecNormalize: Yes

... and 28 more models

✓ Selected model: PPO_returns_20251105_202508
  Type: PPO
  Expected features: 87
  Path: ../checkpoints/PPO_returns_20251105_202508/best_model/best_model.zip

⚠️  IMPORTANT: This model expects 87 features.
   Make sure your data pipeline produces exactly 87 features!


In [4]:
# Load model
if MODEL_PATH is None:
    print("⚠️  Skipping model loading - no models available")
    print("Please train a model first, then re-run this notebook")
    engine = None
else:
    print("Loading model...")
    
    engine = ModelInferenceEngine(
        model_path=MODEL_PATH,
        vecnormalize_path=VECNORMALIZE_PATH,
    )
    
    print(f"✓ Model loaded successfully")
    print(f"  Type: {engine.model_type}")
    print(f"  Has VecNormalize: {engine.vecnormalize is not None}")
    print(f"  Action names: {engine.ACTION_NAMES}")

Loading model...


INFO:rl_trading_lab.live.inference:Loaded PPO model from ../checkpoints/PPO_returns_20251105_202508/best_model/best_model.zip
INFO:rl_trading_lab.live.inference:Loaded VecNormalize from ../checkpoints/PPO_returns_20251105_202508/best_model/vecnormalize.pkl
INFO:rl_trading_lab.live.inference:Initialized ModelInferenceEngine with PPO from best_model.zip


✓ Model loaded successfully
  Type: PPO
  Has VecNormalize: True
  Action names: {0: 'HOLD', 1: 'BUY', 2: 'SELL'}


In [5]:
# Retrieve training configuration
if MODEL_PATH is not None:
    print("Retrieving training configuration...\n")
    
    try:
        config = CheckpointManager.get_training_config(Path(MODEL_PATH))
        
        if config:
            print(f"✓ Configuration retrieved (source: {config.get('source', 'unknown')})\n")
            
            # Display observation config
            if 'observation' in config:
                obs_config = config['observation']
                print("Observation Configuration:")
                print(f"  Input features: {obs_config.get('input_features', [])}")
                print(f"  Validate features: {obs_config.get('validate_features', True)}")
                print()
            
            # Display feature engineering config
            if 'feature_engineering' in config:
                fe_config = config['feature_engineering']
                print("Feature Engineering Configuration:")
                if 'technical_indicators' in fe_config:
                    indicators = fe_config['technical_indicators']
                    if 'sma_ratios' in indicators:
                        print(f"  SMA ratios: {indicators['sma_ratios']}")
                    if 'range_ratios' in indicators:
                        print(f"  Range ratios: {indicators['range_ratios']}")
                    if 'fracdiff' in indicators:
                        print(f"  Fractional differentiation: {indicators['fracdiff']}")
                print()
            
            # Display environment config
            if 'env' in config:
                env_config = config['env']
                if 'environment_params' in env_config:
                    params = env_config['environment_params']
                    print("Environment Configuration:")
                    print(f"  Reward type: {params.get('reward_type', 'unknown')}")
                    print(f"  Initial balance: ${params.get('initial_balance', 10000):,.2f}")
                    print(f"  Commission rate: {params.get('commission_rate', 0.001):.4f}")
                    print(f"  One trade mode: {params.get('one_trade_mode', False)}")
                print()
            
            print("⚠️  IMPORTANT: Ensure your data pipeline matches these configurations!")
            print(f"   Otherwise, the model may produce incorrect predictions.")
        else:
            print("⚠️  Could not retrieve training configuration")
            print("   This may be an older checkpoint without embedded configs.")
            print("   Retrain the model to enable config embedding.")
    
    except Exception as e:
        print(f"⚠️  Error retrieving config: {e}")
        print("   This is expected for checkpoints created before the config fix.")
else:
    print("⚠️  Skipping config retrieval - no model loaded")

INFO:rl_trading_lab.utils.checkpoint_manager:Retrieved training config from embedded metadata


Retrieving training configuration...

✓ Configuration retrieved (source: embedded)

Observation Configuration:
  Input features: ['ratio_sma_5_close_zscore', 'ratio_sma_20_close_zscore', 'ratio_range_close_zscore', 'fracdiff_0.4_zscore']
  Validate features: True

Feature Engineering Configuration:

Environment Configuration:
  Reward type: returns
  Initial balance: $10,000.00
  Commission rate: 0.0000
  One trade mode: False

⚠️  IMPORTANT: Ensure your data pipeline matches these configurations!
   Otherwise, the model may produce incorrect predictions.


In [6]:
# Test model prediction with random data
if engine is not None:
    print("Testing model prediction...\n")
    
    # Create fake features (14 features expected)
    fake_features = np.random.randn(1, 14)
    
    action, confidence = engine.predict(fake_features, deterministic=True)
    action_name = engine.get_action_name(action)
    
    print(f"Action: {action_name} (code: {action})")
    print(f"Confidence: {confidence:.3f}")
    print(f"\n✓ Model prediction working!")
else:
    print("⚠️  Skipping prediction test - no model loaded")

ERROR:rl_trading_lab.live.inference:Error during prediction: 'numpy.ndarray' object has no attribute 'columns'


Testing model prediction...



AttributeError: 'numpy.ndarray' object has no attribute 'columns'

## Part 2: Load Historical Data

Load recent trade data from MinIO to validate the pipeline.

In [ ]:
# Load historical data
print(f"Loading {SYMBOL} data from MinIO...\n")

adapter = BinanceDataAdapter()

# Load last 1 day of data
end_date = datetime.now()
start_date = end_date - timedelta(days=1)

try:
    df = adapter.load_symbol_data(
        symbol=SYMBOL,
        start_date=start_date,
        end_date=end_date,
    )
    
    print(f"✓ Loaded {len(df):,} trades")
    print(f"  Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
    print(f"  Columns: {list(df.columns)}")
    
except Exception as e:
    print(f"✗ Failed to load data: {e}")
    print(f"\nMake sure MinIO is running and data pipeline has been executed.")
    df = None

In [ ]:
# Display sample trades
if df is not None:
    print("Sample trades:\n")
    display(df.head(10))
    
    # Statistics
    print(f"\nStatistics:")
    print(f"  Total volume: {df['quantity'].sum():,.2f} {SYMBOL[:-4]}")
    print(f"  Dollar volume: ${(df['price'] * df['quantity']).sum():,.2f}")
    print(f"  Price range: ${df['price'].min():,.2f} - ${df['price'].max():,.2f}")

## Part 3: Create Dollar Volume Bars

Transform tick data into dollar volume bars (adaptive sampling).

In [ ]:
if df is not None:
    print("Creating dollar volume bars...\n")
    
    # Create bar processor
    bar_processor = BarProcessor(
        dollar_volume_threshold=1_000_000,  # $1M per bar
    )

    # Process trades into bars
    bars_df = bar_processor.process_trades(df)
    
    print(f"✓ Created {len(bars_df)} bars")
    print(f"  Threshold: $1,000,000 per bar")
    print(f"  Time range: {bars_df.index[0]} to {bars_df.index[-1]}")

In [ ]:
# Display sample bars
if df is not None:
    print("Sample bars:\n")
    display(bars_df.head(10))
    
    # Plot bars
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # Price chart
    axes[0].plot(bars_df.index, bars_df['close'], label='Close', linewidth=1)
    axes[0].fill_between(bars_df.index, bars_df['low'], bars_df['high'], alpha=0.2)
    axes[0].set_ylabel('Price (USD)')
    axes[0].set_title(f'{SYMBOL} Dollar Volume Bars (1 day)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Volume chart
    axes[1].bar(bars_df.index, bars_df['volume'], width=0.0001, alpha=0.6)
    axes[1].set_ylabel('Volume')
    axes[1].set_xlabel('Time')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✓ {len(bars_df)} bars created from {len(df):,} trades")

## Part 4: Compute Features

Engineer features that match the training data format.

In [ ]:
if df is not None:
    print("Computing features...\n")
    
    # Create feature pipeline
    pipeline = FeaturePipeline()
    
    # Transform bars into features
    features_df = pipeline.transform(bars_df)
    
    print(f"✓ Computed {len(features_df.columns)} features")
    print(f"  Features: {list(features_df.columns)}")
    print(f"  Shape: {features_df.shape}")

In [ ]:
# Display sample features
if df is not None:
    print("Sample features:\n")
    display(features_df.head(10))
    
    # Feature statistics
    print("\nFeature statistics:")
    display(features_df.describe())

## Part 5: Test Model Predictions

Run the model on historical features to see what actions it would have taken.

In [ ]:
if df is not None:
    print("Testing model predictions on historical data...\n")
    
    # Get predictions for all bars
    predictions = []
    
    for i in range(len(features_df)):
        features = features_df.iloc[i].values.reshape(1, -1)
        action, confidence = engine.predict(features, deterministic=True)
        action_name = engine.get_action_name(action)
        
        predictions.append({
            'bar_index': i,
            'timestamp': bars_df.index[i],
            'price': bars_df.iloc[i]['close'],
            'action': action_name,
            'action_code': action,
            'confidence': confidence,
        })
    
    predictions_df = pd.DataFrame(predictions)
    
    print(f"✓ Generated {len(predictions_df)} predictions")
    print(f"\nAction distribution:")
    print(predictions_df['action'].value_counts())

In [ ]:
# Display predictions
if df is not None:
    print("Sample predictions:\n")
    display(predictions_df.head(20))
    
    # Plot predictions
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Price line
    ax.plot(predictions_df['timestamp'], predictions_df['price'], 
            label='Price', color='black', linewidth=1)
    
    # BUY signals
    buys = predictions_df[predictions_df['action'] == 'BUY']
    ax.scatter(buys['timestamp'], buys['price'], 
              color='green', marker='^', s=100, label='BUY', zorder=5)
    
    # SELL signals
    sells = predictions_df[predictions_df['action'] == 'SELL']
    ax.scatter(sells['timestamp'], sells['price'], 
              color='red', marker='v', s=100, label='SELL', zorder=5)
    
    ax.set_xlabel('Time')
    ax.set_ylabel('Price (USD)')
    ax.set_title(f'{SYMBOL} Model Predictions')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Part 6: Simulate Trading

Simulate trades based on model predictions to estimate performance.

In [ ]:
if df is not None:
    print("Simulating trades...\n")
    
    # Initialize portfolio
    portfolio = PortfolioManager(
        initial_balance=10000,
        symbols=[SYMBOL],
        db_path=":memory:",  # In-memory for simulation
    )
    
    # Execute trades based on predictions
    for _, pred in predictions_df.iterrows():
        if pred['action'] != 'HOLD':
            # Calculate position size ($100 per trade)
            quantity = 100 / pred['price']
            commission = quantity * pred['price'] * 0.001  # 0.1% commission
            
            # Record trade
            portfolio.record_trade(
                symbol=SYMBOL,
                action=pred['action'],
                quantity=quantity,
                price=pred['price'],
                commission=commission,
            )
    
    # Get final statistics
    stats = portfolio.get_stats()
    
    print(f"✓ Simulation complete\n")
    print(f"Results:")
    print(f"  Initial balance: ${stats['initial_balance']:,.2f}")
    print(f"  Final balance: ${stats['total_value']:,.2f}")
    print(f"  Total PnL: ${stats['total_pnl']:+,.2f}")
    print(f"  Returns: {stats['returns']*100:+.2f}%")
    print(f"  Total trades: {stats['total_trades']}")
    print(f"  Commission: ${stats['total_commission']:,.2f}")

In [ ]:
# Analyze trade history
if df is not None:
    trade_history = portfolio.get_trade_history()
    
    if len(trade_history) > 0:
        print("Trade history:\n")
        display(trade_history)
        
        # Calculate additional metrics
        trade_history['pnl'] = trade_history['pnl'].fillna(0)
        
        winning_trades = trade_history[trade_history['pnl'] > 0]
        losing_trades = trade_history[trade_history['pnl'] < 0]
        
        win_rate = len(winning_trades) / len(trade_history) if len(trade_history) > 0 else 0
        avg_win = winning_trades['pnl'].mean() if len(winning_trades) > 0 else 0
        avg_loss = losing_trades['pnl'].mean() if len(losing_trades) > 0 else 0
        
        print(f"\nDetailed Statistics:")
        print(f"  Win rate: {win_rate*100:.1f}%")
        print(f"  Winning trades: {len(winning_trades)}")
        print(f"  Losing trades: {len(losing_trades)}")
        print(f"  Average win: ${avg_win:+,.2f}")
        print(f"  Average loss: ${avg_loss:+,.2f}")
        
        if avg_loss != 0:
            profit_factor = abs(avg_win * len(winning_trades)) / abs(avg_loss * len(losing_trades))
            print(f"  Profit factor: {profit_factor:.2f}")
        
        # Plot cumulative PnL
        trade_history['cumulative_pnl'] = trade_history['pnl'].cumsum()
        
        fig, ax = plt.subplots(figsize=(14, 6))
        ax.plot(range(len(trade_history)), trade_history['cumulative_pnl'], linewidth=2)
        ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
        ax.set_xlabel('Trade Number')
        ax.set_ylabel('Cumulative PnL ($)')
        ax.set_title(f'{SYMBOL} Cumulative PnL (Simulated)')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print("No trades were executed (model only predicted HOLD)")

## Part 7: Setup Safety Guards

Configure risk management and circuit breakers.

In [ ]:
print("Setting up safety guards...\n")

# Create safety guard
safety_guard = SafetyGuard(
    max_drawdown=0.20,  # Stop at 20% drawdown
    max_trades_per_hour=20,
    max_trades_per_day=100,
    initial_balance=10000,
    max_position_pct=0.95,  # Max 95% in positions
    enable_circuit_breaker=True,
)

print(f"✓ Safety guard initialized")
print(f"  Max drawdown: {safety_guard.max_drawdown*100}%")
print(f"  Max trades/hour: {safety_guard.max_trades_per_hour}")
print(f"  Max trades/day: {safety_guard.max_trades_per_day}")
print(f"  Circuit breaker: {'Enabled' if safety_guard.enable_circuit_breaker else 'Disabled'}")

In [ ]:
# Test safety guard
print("Testing safety guard...\n")

# Check if trading is allowed
can_trade, reason = safety_guard.can_trade(SYMBOL)
print(f"Can trade {SYMBOL}: {can_trade}")
if not can_trade:
    print(f"Reason: {reason}")

# Simulate some trades to test rate limiting
print(f"\nSimulating rapid trades...")
for i in range(25):
    can_trade, reason = safety_guard.can_trade(SYMBOL)
    if can_trade:
        # Record a small profit
        safety_guard.record_trade(
            symbol=SYMBOL,
            pnl=10,
            new_balance=10000 + (i+1)*10,
            trade_details={'test': True}
        )
    else:
        print(f"  Trade {i+1}: Blocked - {reason}")
        break

# Get safety guard stats
stats = safety_guard.get_stats()
print(f"\nSafety guard status:")
print(f"  State: {stats['state']}")
print(f"  Current balance: ${stats['current_balance']:,.2f}")
print(f"  Drawdown: {stats['current_drawdown']*100:.1f}%")
print(f"  Violations: {stats['violations']}")

## Part 8: Feature Computer (Real-Time)

Test the real-time feature computation component.

In [ ]:
if df is not None:
    print("Testing real-time feature computer...\n")
    
    # Create feature computer
    feature_computer = FeatureComputer(
        symbol=SYMBOL,
        lookback_window=100,
        min_bars_required=21,  # Need 20 for SMA(20) + 1 for current
    )
    
    print(f"✓ Feature computer initialized")
    print(f"  Lookback window: {feature_computer.lookback_window}")
    print(f"  Min bars required: {feature_computer.min_bars_required}")
    
    # Add bars one by one
    print(f"\nAdding bars...")
    for i in range(min(30, len(bars_df))):
        bar = bars_df.iloc[[i]]  # Keep as DataFrame
        is_ready = feature_computer.add_bar(bar)
        
        if is_ready:
            features = feature_computer.get_latest_features()
            print(f"  Bar {i+1}: Ready! Features shape: {features.shape}")
            break
        else:
            print(f"  Bar {i+1}: Not ready yet (need {feature_computer.min_bars_required})")
    
    # Get prediction
    if is_ready:
        action, confidence = engine.predict(features, deterministic=True)
        action_name = engine.get_action_name(action)
        
        print(f"\n✓ Real-time prediction:")
        print(f"  Action: {action_name}")
        print(f"  Confidence: {confidence:.3f}")

## Part 9: Summary and Next Steps

You've successfully tested all components of the live trading system!

In [ ]:
print("=" * 60)
print("TUTORIAL COMPLETE")
print("=" * 60)
print()
print("✓ Components tested:")
print("  1. Model loading and inference")
print("  2. Historical data loading")
print("  3. Dollar volume bar creation")
print("  4. Feature engineering")
print("  5. Trading simulation")
print("  6. Safety guards")
print("  7. Real-time feature computation")
print()
print("Next steps:")
print("  1. Run validation script:")
print("     uv run python examples/live_trading_example.py validate --model <model_path>")
print()
print("  2. Get Binance testnet API keys:")
print("     https://testnet.binance.vision/")
print()
print("  3. Set environment variables:")
print("     export BINANCE_TESTNET_KEY='your_key'")
print("     export BINANCE_TESTNET_SECRET='your_secret'")
print()
print("  4. Run live trading on testnet:")
print("     uv run python examples/live_trading_example.py trade --model <model_path>")
print()
print("  5. Monitor the dashboard and verify trades")
print()
print("  6. Analyze results:")
print("     uv run python examples/live_trading_example.py analyze --db portfolio.db")
print()
print("IMPORTANT: Always test on testnet before using real money!")
print("=" * 60)

## Appendix: Additional Resources

### Documentation
- `../examples/README.md` - Detailed examples and usage patterns
- `../LIVE_TRADING_GUIDE.md` - Complete system documentation
- `../BINANCE_TESTNET_STATUS.md` - Project status and architecture

### External Links
- [Binance Testnet](https://testnet.binance.vision/) - Get API keys
- [python-binance docs](https://python-binance.readthedocs.io/) - API library
- [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) - RL algorithms

### Safety Reminders
- ⚠️ Testnet uses fake money (no risk)
- ⚠️ Always validate before live trading
- ⚠️ Start with small amounts on live
- ⚠️ Monitor continuously
- ⚠️ Use stop-loss and circuit breakers
- ⚠️ Never invest more than you can afford to lose